In [ ]:
!pip install -q datasets pandas

In [ ]:
from datasets import load_dataset
import pandas as pd

In [ ]:
ds = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")
df = ds["train"].to_pandas()

df.shape
print(df.columns.tolist())
df.head(-1)

['flags', 'instruction', 'category', 'intent', 'response']


,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...
...,...,...,...,...,...
26866,BL,check the current status of my compensation,REFUND,track_refund,I've grasped that you're eager to know the cur...
26867,BL,I am waiting for a rebate of {{Refund Amount}}...,REFUND,track_refund,Thank you for sharing your situation regarding...
26868,BIL,how to see if there is anything wrong with my ...,REFUND,track_refund,Ensuring the accuracy of your restitution is o...
26869,BLQZ,I'm waiting for a reimbjrsement of {{Currency ...,REFUND,track_refund,"Firstly, I genuinely understand the importance..."


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26872 entries, 0 to 26871
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   flags        26872 non-null  object
 1   instruction  26872 non-null  object
 2   category     26872 non-null  object
 3   intent       26872 non-null  object
 4   response     26872 non-null  object
dtypes: object(5)
memory usage: 1.0+ MB


In [8]:
import re

# 1. One full example (nothing truncated)
r = df.iloc[0]
print("CUSTOMER:", r["instruction"])
print("AGENT:", r["response"])
print("-" * 60)

# 2. Lengths (in characters)
df["instr_len"] = df["instruction"].str.len()
df["resp_len"] = df["response"].str.len()
print(df[["instr_len", "resp_len"]].describe(percentiles=[.5, .9, .99]).round(0))

# 3. Placeholders
pat = r"\{\{.*?\}\}"
print("\nInstructions with placeholders:", df["instruction"].str.contains(pat).sum())
print("Responses with placeholders:   ", df["response"].str.contains(pat).sum())
print("\nMost common placeholders in responses:")
print(pd.Series(re.findall(pat, " ".join(df["response"]))).value_counts().head(8))

# 4. Missing / duplicates
print("\nMissing:", df[["instruction", "response"]].isna().sum().sum())
print("Duplicates:", df.duplicated(subset=["instruction", "response"]).sum())

CUSTOMER: question about cancelling order {{Order Number}}
AGENT: I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you.
------------------------------------------------------------
       instr_len  resp_len
count    26872.0   26872.0
mean        47.0     634.0
std         11.0     332.0
min          6.0      57.0
50%         48.0     540.0
90%         59.0    1059.0
99%         71.0    1837.0
max         92.0    2472.0

Instructions with placeholders: 6670
Responses with placeholders:    13006

Most common placeholders in responses:
{{Order Number}}                     5122
{{Account Type}}                     4429
{{Account Category}}                 3078
{{Online Order Interaction}}         2699
{{Customer Support Phone Number}}    2635
{{Website URL}}                      2534
{{Customer Support Hours}}           2325
{{Invoi

In [9]:
import re

d = df[["instruction", "response", "intent", "category"]].copy()

# Collapse stray whitespace/newlines into single spaces
clean = lambda t: re.sub(r"\s+", " ", t).strip()
d["instruction"] = d["instruction"].map(clean)
d["response"] = d["response"].map(clean)

# Drop very long responses
before = len(d)
d = d[d["response"].str.len() <= 1000].reset_index(drop=True)
print(f"Kept {len(d)} of {before} rows ({len(d)/before:.1%})")

# Training format
d["text"] = "Customer: " + d["instruction"] + "\nAgent: " + d["response"] + "<END>\n"
print(d["text"].iloc[0])

# Stats for the tokenizer decision
all_text = "".join(d["text"])
print("Total characters:", len(all_text))
print("Unique characters:", len(set(all_text)))
print("Non-ASCII characters:", sorted({c for c in all_text if ord(c) > 127}))

d.to_csv("clean.csv", index=False)

Kept 23705 of 26872 rows (88.2%)
Customer: question about cancelling order {{Order Number}}
Agent: I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you.<END>

Total characters: 14431045
Unique characters: 109
Non-ASCII characters: ['à', 'é', '–', '—', '’', '✨', '️', '🌟', '💡', '💪', '🔐', '🔒', '🗝', '😊', '🙁', '🙏', '🛡', '🤗', '🤝']


In [11]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/support-gpt', exist_ok=True)
!cp clean.csv /content/drive/MyDrive/support-gpt/

Mounted at /content/drive


In [12]:
!pip install -q tokenizers
import re, numpy as np, pandas as pd
from tokenizers import ByteLevelBPETokenizer

In [13]:
d = pd.read_csv("clean.csv")


train_df = d.sample(frac=0.9, random_state=42)
val_df = d.drop(train_df.index)

In [14]:
placeholders = sorted(set(re.findall(r"\{\{.*?\}\}", "".join(d["text"]))))
print(len(placeholders), "placeholders:", placeholders[:5], "...")


300 placeholders: ['{{Access Key Recovery}}', '{{Access Key}}', '{{Account Category}}', '{{Account Change}}', '{{Account Closure Process}}'] ...


In [15]:
tok = ByteLevelBPETokenizer()

tok.train_from_iterator(train_df["text"], vocab_size=4096, min_frequency=2, special_tokens=["<END>"] + placeholders)

print("Vocab size:", tok.get_vocab_size())

Vocab size: 4096


In [16]:
def encode(frame):
  ids= []

  for t in frame["text"]:
    ids.extend(tok.encode(t).ids)

  return np.array(ids, dtype=np.uint16)

In [17]:
train_ids , val_ids = encode(train_df) , encode(val_df)
print("Train tokens:", len(train_ids), "| Val tokens:", len(val_ids))

Train tokens: 2588333 | Val tokens: 287150


In [18]:
lens = d["text"].map(lambda t: len(tok.encode(t).ids))
print(lens.describe(percentiles=[.5, .9, .99]).round(0))

count    23705.0
mean       121.0
std         39.0
min         25.0
50%        114.0
90%        183.0
99%        217.0
max        255.0
Name: text, dtype: float64


In [20]:
sample = d["text"].iloc[0]
ids = tok.encode(sample).ids
back = tok.decode(ids, skip_special_tokens=False)
print(back == sample)
print(repr(back))


import random
bad = [t for t in random.sample(list(d["text"]), 500)
       if tok.decode(tok.encode(t).ids, skip_special_tokens=False) != t]
print("Mismatches out of 500:", len(bad))

True
"Customer: question about cancelling order {{Order Number}}\nAgent: I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you.<END>\n"
Mismatches out of 500: 0


In [21]:
out = "/content/drive/MyDrive/support-gpt/"
train_ids.tofile(out + "train.bin"); val_ids.tofile(out + "val.bin")
tok.save(out + "tokenizer.json")

In [22]:
!ls -la /content/drive/MyDrive/support-gpt/

total 34335
drwx------ 2 root root     4096 Sep 28 07:47 .
drwx------ 8 root root     4096 Sep 28 07:04 ..
-rw------- 1 root root 29077926 Sep 28 07:04 clean.csv
-rw------- 1 root root   320530 Sep 28 07:47 tokenizer.json
-rw------- 1 root root  5176666 Sep 28 07:47 train.bin
-rw------- 1 root root   574300 Sep 28 07:47 val.bin
